# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [ ]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos


def f(a, b, c):
    return -(a**3) + sin(3 * b) - 1.0 / c + b**2.5 - a**0.5


print(f(2, 3, 4))

6.336362190988558


In [ ]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29


def gradf(a, b, c):
    dfda = -3 * a**2 - 0.5 * a ** (-0.5)
    dfdb = 2.5 * b**1.5 + 3 * cos(3 * b)
    dfdc = c ** (-2)
    return [dfda, dfdb, dfdc]


# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
    ok = "OK" if abs(yours[dim] - ans[dim]) < 1e-5 else "WRONG!"
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [17]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# -----------
a = 2.0
b = 3.0
c = 4.0
h = 0.000001
dfda = (f(a + h, b, c) - f(a, b, c)) / h
dfdb = (f(a, b + h, c) - f(a, b, c)) / h
dfdc = (f(a, b, c + h) - f(a, b, c)) / h
numerical_grad = [dfda, dfdb, dfdc]
# -----------

for dim in range(3):
    ok = "OK" if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else "WRONG!"
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353559348809995
OK for dim 1: expected 10.25699027111255, yours returns 10.256991666679482
OK for dim 2: expected 0.0625, yours returns 0.062499984743169534


In [18]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
a = 2.0
b = 3.0
c = 4.0
h = 0.000001
dfda = (f(a + h, b, c) - f(a - h, b, c)) / (2 * h)
dfdb = (f(a, b + h, c) - f(a, b - h, c)) / (2 * h)
dfdc = (f(a, b, c + h) - f(a, b, c - h)) / (2 * h)
numerical_grad2 = [dfda, dfdb, dfdc]
# -----------

for dim in range(3):
    ok = "OK" if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else "WRONG!"
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553391353245
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027401572
OK for dim 2: expected 0.0625, yours returns 0.06250000028629188


In [25]:
mse1 = sum((numerical_grad[dim] - ans[dim]) ** 2 for dim in range(3))
mse2 = sum((numerical_grad2[dim] - ans[dim]) ** 2 for dim in range(3))
print(f"{mse1=}")
print(f"{mse2=}")

mse1=3.7448186339408513e-11
mse2=9.087922004948263e-18


## section 2: support for softmax

In [110]:
# Value class starter code, with many functions taken out
from math import exp, log


class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):  # exactly as in the video
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    # chain rule: dL/dx = dL/df * df/dx (dL/df = out.grad)
    # f = x * y
    # df/dx = y
    # df/dy = x
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    # f = x^n
    # df/dx = n * x^(n - 1)
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += other * self.data ** (other - 1) * out.grad

        out._backward = _backward
        return out

    # f = e^x
    # df/dx = e^x * 1.0
    def exp(self):
        out = Value(exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    # f = log(x)
    # df/dx = 1/x
    def log(self):
        out = Value(log(self.data), (self,), "log")

        def _backward():
            self.grad += self.data**-1 * out.grad

        out._backward = _backward
        return out

    def backward(self):  # exactly as in video
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self):  # -self
        return self * -1

    def __sub__(self, other):  # self - other
        return self + (-other)

    def __radd__(self, other):  # other + self
        return self + other

    def __truediv__(self, other):  # self / other
        return self * other**-1

    def __rtruediv__(self, other):  # other / self
        return other * self**-1

    def __abs__(self):  # abs(self)
        return abs(self.data)

In [86]:
x = Value(0.000001)
x.log()

Value(data=-13.815510557964274)

In [135]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
    counts = [logit.exp() for logit in logits]
    denominator = sum(counts)
    out = [c / denominator for c in counts]
    return out


# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log()  # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
    ok = "OK" if abs(logits[dim].grad - ans[dim]) < 1e-5 else "WRONG!"
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.8864503806400986


In [139]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch

torch_logits = torch.tensor([0.0, 3.0, -2.0, 1.0], requires_grad=True)
torch_probs = torch.softmax(torch_logits, dim=0)
torch_loss = -torch_probs[3].log()
torch_loss.backward()
print(torch_loss.item())

for dim in range(4):
    torch_grad = torch_logits.grad
    ok = "OK" if abs(logits[dim].grad - torch_grad[dim].item()) < 1e-5 else "WRONG!"
    print(f"{ok} for dim {dim}: torch {torch_grad[dim].item()}, micrograd {logits[dim].grad}")

2.1755154132843018
OK for dim 0: torch 0.041772566735744476, micrograd 0.041772570515350445
OK for dim 1: torch 0.8390244841575623, micrograd 0.8390245074625319
OK for dim 2: torch 0.005653302650898695, micrograd 0.005653302662216329
OK for dim 3: torch -0.8864504098892212, micrograd -0.8864503806400986
